In [1]:
import os
import json
import torch
import pandas as pd
from tqdm import tqdm 
from safetensors.torch import save_file

from vllm import LLM, SamplingParams, ModelRegistry
from models.Qwen3.qwen3_vllm import Qwen3_14B_vLLM
from models.Qwen3.qwen3 import Qwen3Tokenizer, QWEN_14B_CFG

/home/simpsonadmin/anaconda3/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 05-13 05:30:48 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-13 05:30:48 [nixl_utils.py:34] NIXL is not available
WARNING 05-13 05:30:48 [nixl_utils.py:44] NIXL agent config is not available




### Step 0: Translate Pretrained to vLLM-Safetensor 

Convert a custom Qwen3-14B `.pth` checkpoint into a vLLM-compatible format by:

1. translate_pretrained_to_vLLM: Remaps custom state_dict keys (e.g., `tok_emb.weight`, `transformer_blocks.*.att.W_query.weight`) to the HuggingFace/vLLM naming convention (e.g., `model.embed_tokens.weight`, `model.layers.*.self_attn.q_proj.weight`).

2. convert_pth_to_vllm:
    - Loads the merged `.pth` state_dict
    - Translates all keys and saves as `model.safetensors`
    - Generates a `config.json` with architecture metadata (hidden size, num layers, heads, RoPE config, etc.) and sets `"architectures": ["Qwen3vLLM"]`
    - Copies the tokenizer via `Qwen3Tokenizer` into the output directory

In [2]:
def translate_pretrained_to_vLLM(name: str) -> str:
    """Translate qwen3.py state_dict keys to Qwen3-vLLM keys."""
    if name.startswith("model.") or name.startswith("lm_head."):
        return name
    if name == "tok_emb.weight":
        return "model.embed_tokens.weight"
    if name == "final_norm.scale":
        return "model.norm.weight"
    if name == "out_head.weight":
        return "lm_head.weight"

    if name.startswith("transformer_blocks."):
        parts = name.split(".")
        layer_idx = parts[1]
        rest = ".".join(parts[2:])
        rest = rest.replace("att.W_query.weight", "self_attn.q_proj.weight")
        rest = rest.replace("att.W_key.weight", "self_attn.k_proj.weight")
        rest = rest.replace("att.W_value.weight", "self_attn.v_proj.weight")
        rest = rest.replace("att.out_proj.weight", "self_attn.o_proj.weight")
        rest = rest.replace("att.q_norm.scale", "self_attn.q_norm.weight")
        rest = rest.replace("att.k_norm.scale", "self_attn.k_norm.weight")
        rest = rest.replace("norm1.scale", "input_layernorm.weight")
        rest = rest.replace("norm2.scale", "post_attention_layernorm.weight")
        rest = rest.replace("ff.fc1.weight", "mlp.gate_proj.weight")
        rest = rest.replace("ff.fc2.weight", "mlp.up_proj.weight")
        rest = rest.replace("ff.fc3.weight", "mlp.down_proj.weight")
        return f"model.layers.{layer_idx}.{rest}"
    return name


def convert_pth_to_vllm(tokenizer = None, model_cfg = None,  pth_path: str = None, output_dir: str = None ):
    """Convert a merged .pth checkpoint to vLLM-compatible safetensors format.
    Args:
        pth_path: Path to the merged .pth state_dict file.
        output_dir: Output folder to save model.safetensors, config.json, tokenizer.json.
        model_cfg: Config dataclass instance (e.g. QWEN_14B_CFG).
        tokenizer: Tokenizer loaded from the model base.
    """

    # 0. Check the files is exist to skip the processing 
    required_files = ["model.safetensors", "config.json", "tokenizer.json"]
    missing = [f for f in required_files if not os.path.exists(os.path.join(output_dir, f))]
    if not missing:
        print(f"All files exist in {output_dir} → skipping conversion.")
        return
    print(f"Missing files: {missing} → running conversion ...")
    os.makedirs(output_dir, exist_ok=True)

    # 1. Load .pth file 
    state_dict = torch.load(pth_path, map_location="cpu", weights_only=True)
    print(f"Loaded: {pth_path}")

    # 2. Translate and save safetensors vLLM
    hf_state_dict = {}
    for key, tensor in state_dict.items():
        hf_state_dict[translate_pretrained_to_vLLM(key)] = tensor
    safetensors_path = os.path.join(output_dir, "model.safetensors")
    save_file(hf_state_dict, safetensors_path)
    print(f"Saved: {safetensors_path}")

    # 3. config.json
    config = {
        "architectures": ["Qwen3vLLM"],
        "model_type": "qwen3",
        "vocab_size": model_cfg.vocab_size,
        "hidden_size": model_cfg.emb_dim,
        "intermediate_size": model_cfg.hidden_dim,
        "num_hidden_layers": model_cfg.n_blocks,
        "num_attention_heads": model_cfg.n_heads,
        "num_key_value_heads": model_cfg.n_kv_groups,
        "head_dim": model_cfg.head_dim,
        "max_position_embeddings": model_cfg.context_length,
        "rope_theta": model_cfg.rope_base,
        "tie_word_embeddings": False,
        "torch_dtype": "bfloat16",
    }
    config_path = os.path.join(output_dir, "config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Saved: {config_path}")

    # 4. save tokenizer.json
    tok_output = os.path.join(output_dir, "tokenizer.json")
    tokenizer._tok.save(tok_output)
    print(f"Saved: {tok_output}")

In [3]:
# It takes long time to save safetensor - config
convert_pth_to_vllm(
    tokenizer=Qwen3Tokenizer("./models/Qwen3/tokenizer.json"),
    model_cfg=QWEN_14B_CFG,
    pth_path="./logs/9_Qwen3_14B_Jigsaw_LoRA_r16_a32/11_05_2026/version_3/model_pretrained/00-0.2356-0.8897.pth",
    output_dir="./model_vllm",
)

All files exist in ./model_vllm → skipping conversion.


### Step 1: Register vLLM

In [4]:
# ---- Configuration ----
# SAFETENSOR_DIR: model.safetensors + config.json (+ tokenizer.json)
# config.json must have: "architectures": ["Qwen3vLLM"] --> please refer convert_pth_to_vllm
SAFETENSOR_DIR = "./logs/9_Qwen3_14B_Jigsaw_LoRA_r16_a32/11_05_2026/version_3/model_vllm"
ModelRegistry.register_model("Qwen3vLLM", Qwen3_14B_vLLM)
print(f"Loaded model from: {SAFETENSOR_DIR}")

Loaded model from: ./logs/9_Qwen3_14B_Jigsaw_LoRA_r16_a32/11_05_2026/version_3/model_vllm


### Step 2: Launch Engine vLLM 

In [5]:
llm = LLM(
    model=SAFETENSOR_DIR,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.85,
)
print("vLLM engine ready")

INFO 05-13 05:30:48 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': './logs/9_Qwen3_14B_Jigsaw_LoRA_r16_a32/11_05_2026/version_3/model_vllm'}
INFO 05-13 05:30:48 [model.py:555] Resolved architecture: Qwen3vLLM
INFO 05-13 05:30:48 [model.py:1680] Using max model len 40960
INFO 05-13 05:30:48 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-13 05:30:48 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-13 05:30:48 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


[transformers] The tokenizer you are loading from './logs/9_Qwen3_14B_Jigsaw_LoRA_r16_a32/11_05_2026/version_3/model_vllm' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore pid=2021405) INFO 05-13 05:30:49 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='./logs/9_Qwen3_14B_Jigsaw_LoRA_r16_a32/11_05_2026/version_3/model_vllm', speculative_config=None, tokenizer='./logs/9_Qwen3_14B_Jigsaw_LoRA_r16_a32/11_05_2026/version_3/model_vllm', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=Fals

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:29<00:00, 29.85s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:29<00:00, 29.86s/it]
(EngineCore pid=2021405) 


(EngineCore pid=2021405) INFO 05-13 05:31:21 [default_loader.py:384] Loading weights took 30.06 seconds
(EngineCore pid=2021405) INFO 05-13 05:31:22 [gpu_model_runner.py:4879] Model loading took 27.52 GiB memory and 30.812282 seconds
(EngineCore pid=2021405) INFO 05-13 05:31:25 [gpu_model_runner.py:5963] Profiling CUDA graph memory: PIECEWISE=51 (largest=512), FULL=35 (largest=256)
(EngineCore pid=2021405) INFO 05-13 05:31:26 [gpu_model_runner.py:6042] Estimated CUDA graph memory: 0.76 GiB total
(EngineCore pid=2021405) INFO 05-13 05:31:26 [gpu_worker.py:440] Available KV cache memory: 37.73 GiB
(EngineCore pid=2021405) INFO 05-13 05:31:26 [gpu_worker.py:455] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.8500 is equivalent to --gpu-memory-utilization=0.8404 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8596. To disable, set VLLM_MEMORY_PROFILER_

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:06<00:00,  8.12it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:04<00:00,  7.86it/s]


(EngineCore pid=2021405) INFO 05-13 05:31:38 [gpu_model_runner.py:6133] Graph capturing finished in 11 secs, took 0.69 GiB
(EngineCore pid=2021405) INFO 05-13 05:31:38 [gpu_worker.py:599] CUDA graph pool memory: 0.69 GiB (actual), 0.76 GiB (estimated), difference: 0.07 GiB (9.9%).
(EngineCore pid=2021405) INFO 05-13 05:31:38 [core.py:306] init engine (profile, create kv cache, warmup model) took 15.96 s
(EngineCore pid=2021405) INFO 05-13 05:31:38 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
vLLM engine ready


### Step 3: Inference on Jigsaw test dataset

In [10]:
test_df = pd.read_csv("./data/Jigsaw2026/test.csv")
sample = test_df.iloc[0]

BASE_PROMPT = "Reddit moderation: Does the comment violate the rule? Answer 'Yes' or 'No' only."
prompt = (
    f"<|im_start|>system\n{BASE_PROMPT}<|im_end|>\n"
    f"<|im_start|>user\nComment: {sample['body']}\n\nrule: {sample['rule']}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=3,
    stop_token_ids=[151643, 151645],  # eos_token, <|im_end|>
)

outputs = llm.generate([prompt], sampling_params)
generated = outputs[0].outputs[0].text.strip()

print(f"\n{'='*60}")
print(f"Test row_id: {sample['row_id']}")
print(f"Body: {sample['body'][:200]}...")
print(f"Rule: {sample['rule']}")
print(f"Prediction: {generated}")
print(f"{'='*60}")

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 11.56it/s, est. speed input: 977.96 toks/s, output: 23.28 toks/s]


Test row_id: 2029
Body: NEW RAP GROUP 17. CHECK US OUT https://soundcloud.com/user-125895482...
Rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.
Prediction: No


In [6]:
test_df = pd.read_csv("./data/Jigsaw2026/train.csv")
sample = test_df.iloc[0]

BASE_PROMPT = "Reddit moderation: Does the comment violate the rule? Answer 'Yes' or 'No' only."
prompt = (
    f"<|im_start|>system\n{BASE_PROMPT}<|im_end|>\n"
    f"<|im_start|>user\nComment: {sample['body']}\n\nrule: {sample['rule']}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=8,
    stop_token_ids=[151643, 151645],  # eos_token, <|im_end|>
)

outputs = llm.generate([prompt], sampling_params)
generated = outputs[0].outputs[0].text.strip()

print(f"\n{'='*60}")
print(f"Test row_id: {sample['row_id']}")
print(f"Body: {sample['body'][:200]}...")
print(f"Rule: {sample['rule']}")
print(f"Prediction: {generated}")
print(f"{'='*60}")

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 11.09it/s, est. speed input: 796.47 toks/s, output: 22.42 toks/s]


Test row_id: 0
Body: Banks don't want you to know this! Click here to know more!...
Rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.
Prediction: No
